In [0]:
"""
03_gold_datamart.py

Purpose:
--------
This notebook implements the Gold layer of the Medallion Architecture.

The notebook performs:
- Create gold_top_item datamart
- Calculate yearly views
- Rank items
- Determine most used platform

Execution Order:
----------------
1. 01_bronze_ingestion
2. 02_silver_transformation
3. 03_gold_datamart

Assumptions:
------------
- event_type = 'view_item' represents item views
- parameter_name = 'item_id' contains valid item identifiers
- year column exists in Silver layer
"""

# --------------------------------------------------
# Import Required Libraries
# --------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# Read Silver Tables
# --------------------------------------------------

event_df = spark.table("silver_event")
item_df = spark.table("silver_item")

# --------------------------------------------------
# DDL Statements
# --------------------------------------------------
# Explicit DDLs are included to satisfy
# assignment requirements.
# --------------------------------------------------

spark.sql("""
    CREATE TABLE IF NOT EXISTS gold_top_item (
        year INT,
        item_id STRING,
        total_views BIGINT,
        item_rank INT,
        most_used_platform STRING
    )
    USING DELTA
    PARTITIONED BY (year)
""")

# ============================================================
# top_item datamart will be created after performing below steps

# ============================================================
# STEP 1: Filter Valid Item View Events
# ============================================================
# Only rows satisfying below conditions are included:
#
# - event_type = view_item
# - parameter_name = item_id
#
# This avoids inclusion of:
# - page navigation events
# - search events
# - wishlist events
# - non-item activities
# ============================================================

view_events_df = (
    event_df
    .filter(
        (F.col("event_type") == "view_item") &
        (F.col("parameter_name") == "item_id")
    )
)

# ============================================================
# STEP 2: Calculate Platform Views
# ============================================================
# Platform level aggregation is used to determine:
# - most used platform
# - yearly platform activity
# ============================================================

platform_views_df = (
    view_events_df
    .groupBy(
        "year",
        "item_id",
        "platform"
    )
    .agg(
        F.count("*").alias(
            "platform_view_count"
        )
    )
)

# ============================================================
# STEP 3: Calculate Total Item Views
# ============================================================
# Calculates total number of views
# for every:
# - item
# - year
# ============================================================

total_views_df = (
    platform_views_df
    .groupBy(
        "year",
        "item_id"
    )
    .agg(
        F.sum(
            "platform_view_count"
        ).alias("total_views")
    )
)

# ============================================================
# STEP 4: Rank Items By Total Views
# ============================================================
# Spark window functions are used to rank
# items within each year based on:
# - descending total views
# ============================================================

rank_window = Window.partitionBy("year").orderBy(F.col("total_views").desc())

ranked_items_df = (
    total_views_df
    .withColumn(
        "item_rank",
        F.rank().over(rank_window)
    )
)

# ============================================================
# STEP 5: Identify Most Used Platform
# ============================================================
# row_number() is used to determine
# the most frequently used platform
# for each item/year combination.
# ============================================================

platform_window = Window.partitionBy(
    "year",
    "item_id"
).orderBy(
    F.col("platform_view_count").desc()
)

most_used_platform_df = (
    platform_views_df
    .withColumn(
        "rn",
        F.row_number().over(platform_window)
    )
    .filter(
        F.col("rn") == 1
    )
    .select(
        "year",
        "item_id",
        F.col("platform").alias(
            "most_used_platform"
        )
    )
)

# ============================================================
# STEP 6: Create Final top_item Datamart
# ============================================================
# Final Gold datamart contains:
#
# - item details
# - yearly item views
# - item ranking
# - most used platform
# ============================================================

top_item_datamart_df = (
    ranked_items_df
    .join(
        most_used_platform_df,
        ["year", "item_id"],
        "left"
    )
    .join(
        item_df,
        ranked_items_df.item_id == item_df.id,
        "left"
    )
    .select(
        ranked_items_df.year,
        ranked_items_df.item_id,
        F.col("name").alias(
            "item_name"
        ),
        F.col("category"),
        F.col("total_views"),
        F.col("item_rank"),
        F.col("most_used_platform")
    )

)

# --------------------------------------------------
# Validate Gold Datamart
# --------------------------------------------------

print("Gold Datamart Output:")

top_item_datamart_df.show(100, truncate=False)

print("Gold Datamart Record Count:")

print(top_item_datamart_df.count())

# --------------------------------------------------
# Save Gold Delta Table
# --------------------------------------------------
# overwriteSchema is used to support
# schema evolution during development.
# --------------------------------------------------

top_item_datamart_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year") \
    .saveAsTable("gold_top_item")

# --------------------------------------------------
# Gold Layer Completion
# --------------------------------------------------

print("Gold datamart created successfully.")

print("Table Created:")

print("- gold_top_item")
